# MTC Smart Challenge 2026 — Entrenamiento Baseline (Fase 2.1)

Notebook de entrenamiento end-to-end para Kaggle (P100 / T4 / T4×2).

**Qué hace este notebook:**
1. Carga `src/` y los manifiestos del split desde el repo (git clone) o un dataset adjunto.
2. Descubre `train.csv` y los frames en `/kaggle/input/mtc-smart-challenge-trai/`.
3. Convierte `train.csv` → labels YOLO-OBB 4-esquinas en `/kaggle/working/labels/` (~27 MB).
4. Reconstruye el split exacto desde `configs/train_ids.txt` / `val_ids.txt` (sin re-aleatorizar).
5. Crea la estructura YOLO con **symlinks** (frames en `/kaggle/input/`; 0 bytes copiados).
6. Entrena **yolo11s-obb** baseline: imgsz=1024, epochs=25, batch=auto, seed=42.
7. Imprime min/época tras la primera para estimar si el entrenamiento cabe en 9 h.
8. Guarda `best.pt` y `last.pt` como outputs del notebook.

**Uso de disco en /kaggle/working (límite 20 GB):**
- Labels convertidos: ~27 MB
- Repo clonado: ~15 MB
- Checkpoints YOLO cada 5 épocas: ~150 MB total
- Los 40 GB de frames permanecen en `/kaggle/input/` (no cuentan)

**Para retomar una sesión interrumpida:**  
En `cell-train` cambiar `MODEL` a la ruta del último checkpoint y añadir `resume=True`.

In [ ]:
# ── Celda 1: Entorno — GPU, dependencias, clone ───────────────────────────
import subprocess, sys, os
from pathlib import Path

# Instalar ultralytics (incluye PyTorch si no está; en Kaggle ya viene)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "ultralytics>=8.3"])

import torch

n_gpu  = torch.cuda.device_count()
device = 0 if n_gpu == 1 else list(range(n_gpu))
print(f"PyTorch {torch.__version__}  |  GPUs disponibles: {n_gpu}  |  device: {device}")
for i in range(n_gpu):
    p  = torch.cuda.get_device_properties(i)
    gb = p.total_memory / 1e9
    print(f"  GPU {i}: {p.name}  {gb:.1f} GB VRAM")

In [ ]:
# ── Celda 2: Cargar repo (src/ + manifiestos del split) ───────────────────
#
# ── OPCIÓN A (activa): git clone desde repo público ───────────────────────
#    Prerequisito: commit + push de configs/ y src/ antes de correr en Kaggle:
#      git add configs/ src/ notebooks/
#      git commit -m "feat: kaggle training notebook + split manifests"
#      git push
#    Luego habilitar internet en Settings del notebook.
#
# ── OPCIÓN B: dataset de código adjunto (sin repo público) ────────────────
#    1. Crea un dataset en Kaggle con: src/, configs/train_ids.txt, val_ids.txt
#    2. Adjúntalo al notebook (Settings > Add data)
#    3. Comenta el bloque OPCIÓN A y descomenta OPCIÓN B:
#
# REPO_DIR = Path("/kaggle/input/<slug-de-tu-dataset-de-codigo>")
# if str(REPO_DIR) not in sys.path:
#     sys.path.insert(0, str(REPO_DIR))
# for f in ["configs/train_ids.txt", "configs/val_ids.txt"]:
#     p = REPO_DIR / f
#     assert p.exists(), f"Falta {p} en el dataset de codigo"
#     print(f"  {f}: {sum(1 for _ in p.open())} IDs")
# print("src/ cargado desde dataset adjunto.")

# ── OPCIÓN A ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/HaroldVRY/MTC-Challenge.git"
REPO_DIR = Path("/kaggle/working/repo")

if not REPO_DIR.exists():
    print(f"Clonando {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth=1", REPO_URL, str(REPO_DIR)])
else:
    print(f"Repo ya presente en {REPO_DIR}; actualizando ...")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Verificar que los manifiestos del split están en el repo (commiteados)
for f in ["configs/train_ids.txt", "configs/val_ids.txt"]:
    p = REPO_DIR / f
    assert p.exists(), f"Falta {p} — commitea configs/ al repo antes de correr."
    print(f"  {f}: {sum(1 for _ in p.open())} IDs")

print("Repo listo.")

In [ ]:
# ── Celda 3: Descubrimiento de rutas del dataset ──────────────────────────
#
# COMP: slug del dataset de Kaggle con los frames de entrenamiento.
#   → Ajusta si el nombre difiere (Settings > Data > ver el nombre exacto).
# Los frames NO se copian a /kaggle/working (~40 GB excede el límite de 20 GB).
# Se usan symlinks: yolo_data/images/ → COMP  (0 bytes copiados).

COMP  = Path("/kaggle/input/mtc-smart-challenge-trai")
WORK  = Path("/kaggle/working")

# train.csv: buscar en COMP; si no está, ampliar búsqueda a todo /kaggle/input
train_csv_candidates = list(COMP.rglob("train.csv"))
if not train_csv_candidates:
    train_csv_candidates = list(Path("/kaggle/input").rglob("train.csv"))
assert train_csv_candidates, (
    "No se encontró train.csv en ningún dataset adjunto. "
    "Asegúrate de que el dataset que lo contiene está añadido al notebook."
)
TRAIN_CSV = train_csv_candidates[0]
print(f"train.csv   : {TRAIN_CSV}")

# Directorio de frames: el primer .jpg localiza la carpeta raíz
jpg_sample = next(COMP.rglob("*.jpg"), None)
assert jpg_sample is not None, (
    f"No se encontraron .jpg bajo {COMP}. "
    "Verifica que COMP apunta al slug correcto del dataset de frames."
)
FRAMES_DIR = jpg_sample.parent
n_frames   = sum(1 for _ in FRAMES_DIR.glob("*.jpg"))
print(f"frames_dir  : {FRAMES_DIR}")
print(f"frames count: {n_frames:,} JPGs")
assert n_frames >= 50000, f"Se esperaban >=50,000 frames, encontrados {n_frames:,}"

In [ ]:
# ── Celda 4: Convertir train.csv → labels YOLO-OBB 4-esquinas (idempotente) ─
#
# Genera un .txt por frame con formato:
#   class_id x1 y1 x2 y2 x3 y3 x4 y4  (coordenadas normalizadas [0,1])
# El script filtra cajas degeneradas (w<1 o h<1) y clampea esquinas fuera de imagen.
# Tiempo estimado: ~2-3 min para 54,262 frames.

LABELS_DIR = WORK / "labels"
n_existing = sum(1 for _ in LABELS_DIR.glob("*.txt")) if LABELS_DIR.exists() else 0

if n_existing >= 54000:
    print(f"Labels ya convertidos: {n_existing:,} archivos en {LABELS_DIR}")
else:
    from src.convert_to_yolo_obb import convert
    print(f"Convirtiendo {TRAIN_CSV} → {LABELS_DIR} ...")
    convert(TRAIN_CSV, LABELS_DIR)
    n_labels = sum(1 for _ in LABELS_DIR.glob("*.txt"))
    print(f"Listo: {n_labels:,} archivos .txt generados.")

In [ ]:
# ── Celda 5: Reconstruir split y crear estructura YOLO con symlinks ────────
#
# Usa exactamente los IDs de configs/train_ids.txt y configs/val_ids.txt
# que fueron commiteados al repo (split greedy 85/15, agrupado por video_id,
# seed=42, min 15 instancias por clase en val).  NO re-aleatoriza.
#
# Estructura resultado:
#   yolo_data/
#     images/train/<frame_id>.jpg  →  symlink a /kaggle/input/.../<frame_id>.jpg
#     images/val/<frame_id>.jpg    →  symlink a /kaggle/input/.../<frame_id>.jpg
#     labels/train/<frame_id>.txt  →  symlink a /kaggle/working/labels/<frame_id>.txt
#     labels/val/<frame_id>.txt    →  symlink a /kaggle/working/labels/<frame_id>.txt
#
# Tiempo: ~30 s para 54k symlinks.

train_ids = (REPO_DIR / "configs/train_ids.txt").read_text().strip().splitlines()
val_ids   = (REPO_DIR / "configs/val_ids.txt").read_text().strip().splitlines()
print(f"Split cargado:  train={len(train_ids):,}  val={len(val_ids):,}")

YOLO_DIR = WORK / "yolo_data"

for split, ids in [("train", train_ids), ("val", val_ids)]:
    img_dir = YOLO_DIR / "images" / split
    lbl_dir = YOLO_DIR / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    n_img_new = n_lbl_new = n_miss = 0
    for fid in ids:
        src_img = FRAMES_DIR / f"{fid}.jpg"
        src_lbl = LABELS_DIR / f"{fid}.txt"
        dst_img = img_dir    / f"{fid}.jpg"
        dst_lbl = lbl_dir    / f"{fid}.txt"

        if not dst_img.exists():
            if src_img.exists():
                os.symlink(src_img, dst_img)
                n_img_new += 1
            else:
                n_miss += 1

        if not dst_lbl.exists() and src_lbl.exists():
            os.symlink(src_lbl, dst_lbl)
            n_lbl_new += 1

    status = f"{split:5s}: {n_img_new:,} img-symlinks nuevos  |  {n_lbl_new:,} lbl-symlinks nuevos"
    if n_miss:
        status += f"  |  {n_miss} imgs NO encontradas  ← revisar FRAMES_DIR"
    print(f"  {status}")

print(f"Estructura YOLO lista en {YOLO_DIR}")

In [ ]:
# ── Celda 6: Generar data.yaml con rutas de Kaggle ────────────────────────
#
# El configs/data.yaml del repo tiene rutas de Windows (C:/Users/...) que no
# sirven en Kaggle.  Generamos uno nuevo apuntando a /kaggle/working/yolo_data.

import yaml

data_cfg = {
    "path":  str(YOLO_DIR),
    "train": "images/train",
    "val":   "images/val",
    "nc":    9,
    "names": [
        "auto", "combi", "microbus", "minibus", "omnibus",
        "articulado", "camion", "mototaxi", "motocicleta",
    ],
}

DATA_YAML = WORK / "data.yaml"
with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print("data.yaml generado:")
print(DATA_YAML.read_text())

In [ ]:
# ── Celda 7: Configuración del baseline y callbacks de tiempo ─────────────
#
# Parámetros del baseline (Fase 2.1):
#   Modelo  : yolo11s-obb  (small, ~9 M params, punto de partida económico)
#   imgsz   : 1024  (buen compromiso entre resolución y velocidad en T4)
#   epochs  : 25   (suficiente para validar la pipeline; la final irá a 100)
#   batch   : -1   (auto: ultralytics ajusta al ~60% de VRAM disponible)
#   device  : 0 si 1 GPU, [0,1] si T4×2
#   patience: 15   (early stopping; si val no mejora en 15 épocas, para)

import time
from ultralytics import YOLO

SEED    = 42
MODEL   = "yolo11s-obb.pt"
EPOCHS  = 25
IMGSZ   = 1024
WORKERS = 2  # Kaggle kernels tienen CPU limitada

n_gpu  = torch.cuda.device_count()
device = 0 if n_gpu == 1 else list(range(n_gpu))
print(f"device: {device}  |  model: {MODEL}  |  imgsz: {IMGSZ}  |  epochs: {EPOCHS}")

# Callbacks para monitorear el tiempo por época
_t_start  = [None]
_ep_times = []

def _on_epoch_start(trainer):
    _t_start[0] = time.time()

def _on_epoch_end(trainer):
    elapsed = time.time() - _t_start[0]
    _ep_times.append(elapsed)
    ep  = trainer.epoch + 1
    avg = sum(_ep_times) / len(_ep_times)
    rem = (EPOCHS - ep) * avg
    h, m = divmod(int(rem), 3600)
    # Extraer mAP50 de las métricas del trainer si ya está disponible
    metrics = getattr(trainer, "metrics", {}) or {}
    map50   = metrics.get("metrics/mAP50(B)", float("nan"))
    print(
        f"  Época {ep:>3}/{EPOCHS}  "
        f"{elapsed/60:>5.1f} min/época  |  "
        f"restante ~{h}h{m:02d}m  |  "
        f"mAP50={map50:.4f}"
    )
    # Aviso si el total estimado supera la sesión de 9h
    if ep == 1 and EPOCHS * elapsed > 9 * 3600:
        print(
            f"  *** AVISO: tiempo total estimado {EPOCHS*elapsed/3600:.1f} h  "
            f"supera la sesión de 9h. Reduce epochs o imgsz. ***"
        )

print("Callbacks definidos.")

In [ ]:
# ── Celda 8: Entrenamiento baseline ──────────────────────────────────────
#
# Checkpoints cada 5 épocas (save_period=5) para retomar sesiones interrumpidas.
# Para reanudar: reemplaza MODEL por la ruta al último checkpoint .pt y añade
#   resume=True
# en el llamado de abajo.
#
# Augmentations activas (builtin de Ultralytics):
#   mosaic=1.0, fliph=0.5, HSV, degrees, scale — suficiente para baseline.
#   mixup y copy_paste desactivados para reducir tiempo por época.

torch.manual_seed(SEED)

model = YOLO(MODEL)
model.add_callback("on_train_epoch_start", _on_epoch_start)
model.add_callback("on_train_epoch_end",   _on_epoch_end)

print(f"Iniciando entrenamiento | {MODEL} | imgsz={IMGSZ} | epochs={EPOCHS}")
print("-" * 60)

results = model.train(
    data      = str(DATA_YAML),
    epochs    = EPOCHS,
    imgsz     = IMGSZ,
    batch     = -1,          # auto-batch: ~60% VRAM
    device    = device,
    workers   = WORKERS,
    seed      = SEED,
    # ── Optimizador ──────────────────────────────────────────────────
    optimizer     = "AdamW",
    lr0           = 0.001,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    # ── Augmentation ─────────────────────────────────────────────────
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 10.0,
    translate  = 0.1,
    scale      = 0.5,
    fliplr     = 0.5,
    flipud     = 0.0,
    mosaic     = 1.0,
    mixup      = 0.0,    # desactivado en baseline para velocidad
    copy_paste = 0.0,    # ídem
    # ── Entrenamiento ────────────────────────────────────────────────
    patience    = 15,
    save        = True,
    save_period = 5,    # checkpoint cada 5 épocas para retomar sesiones
    # ── Salida ───────────────────────────────────────────────────────
    project  = str(WORK / "runs"),
    name     = "baseline_s",
    exist_ok = True,
    val      = True,
    plots    = True,
    verbose  = False,   # silencia el log por lote; los callbacks ya muestran resumen
)

In [ ]:
# ── Celda 9: Copiar pesos y resumen final ─────────────────────────────────
#
# best.pt y last.pt se copian a /kaggle/working/ para que Kaggle los registre
# como outputs del notebook y estén disponibles en la próxima sesión.

import shutil

run_dir  = Path(results.save_dir)
best_src = run_dir / "weights" / "best.pt"
last_src = run_dir / "weights" / "last.pt"

for src, dst_name in [(best_src, "best.pt"), (last_src, "last.pt")]:
    dst = WORK / dst_name
    if src.exists():
        shutil.copy(src, dst)
        print(f"Guardado: {dst}  ({dst.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"No encontrado: {src}")

# ── Resumen de tiempos ────────────────────────────────────────────────────
print()
print("=" * 55)
print("RESUMEN DEL ENTRENAMIENTO")
print("=" * 55)
if _ep_times:
    avg_min   = sum(_ep_times) / len(_ep_times) / 60
    total_h   = sum(_ep_times) / 3600
    print(f"  Épocas completadas    : {len(_ep_times)}/{EPOCHS}")
    print(f"  Tiempo por época (avg): {avg_min:.1f} min")
    print(f"  Tiempo total          : {total_h:.2f} h")
    print(f"  Estimado 100 épocas   : {avg_min*100/60:.1f} h")
print(f"  Run dir               : {run_dir}")
print("=" * 55)

print()
print("Archivos .pt en /kaggle/working/:")
for f in sorted(WORK.glob("*.pt")):
    print(f"  {f.name:<12}  {f.stat().st_size/1e6:.1f} MB")

print()
print("Siguiente paso: Prompt 2.2 — inferencia sobre test set y generación de submission.csv")